# Vision LLM đọc hoá đơn: JSON có cấu trúc, đo từng trường

Bạn cho `qwen3-vl:2b` (Ollama) đọc 100 hoá đơn của tập test `cord-v2`, trả JSON gồm các dòng hàng và tổng tiền, rồi chấm
bằng bộ chấm **không dùng LLM** của bài Basic 02. Sau đó đổi độ phân giải ảnh (số token ảnh), và so với pipeline hai bước
"OCR rồi LLM văn bản" (`qwen2.5:3b`).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, hằng. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, gọi Ollama, in kết quả; test bỏ qua chúng.
- Cần Ollama với `qwen3-vl:2b` và `qwen2.5:3b`. Hai model không cần nằm cùng lúc trong VRAM: notebook chạy hết phần
  vision rồi mới tới phần văn bản, và giải phóng model sau mỗi phần.
- Chấm: `uv run pytest tracks/ai-engineer/modules/18-multimodal/exercises/intermediate/01-receipt-vlm -q`

In [ ]:
# explore: tìm gốc repo, tải cord-v2 nếu chưa có
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

CORD = ensure("cord-v2")
CACHE = CORD / "cache"
CACHE.mkdir(exist_ok=True)
print(CORD, sorted(p.name for p in (CORD / "data").iterdir()))

In [ ]:
import base64
import difflib
import io
import json
import re
import time

from PIL import Image

## 1. Bộ chấm (dán từ Basic 02)

Notebook tự chứa, không import chéo bài: **dán** sáu hàm của bài Basic 02 (`parse_amount`, `gold_from_cord`,
`name_similarity`, `match_items`, `score_receipt`, `aggregate`) vào cell dưới, kèm các hàm phụ bạn đã viết. Test kiểm
nhanh lại chúng.

In [ ]:
def parse_amount(text) -> float | None:
    # TODO: dán từ Basic 02
    raise NotImplementedError


def gold_from_cord(gt_parse: dict) -> dict:
    # TODO: dán từ Basic 02
    raise NotImplementedError


def name_similarity(a: str, b: str) -> float:
    # TODO: dán từ Basic 02
    raise NotImplementedError


def match_items(pred: list[dict], gold: list[dict], min_name_sim: float = 0.5) -> list[tuple[int, int]]:
    # TODO: dán từ Basic 02
    raise NotImplementedError


def score_receipt(pred: dict | None, gold: dict) -> dict:
    # TODO: dán từ Basic 02
    raise NotImplementedError


def aggregate(scores: list[dict]) -> dict:
    # TODO: dán từ Basic 02
    raise NotImplementedError

## 2. Ảnh, message và đọc câu trả lời

`qwen3-vl:2b` trên Ollama 0.13.5 là bản **thinking**: nó viết một đoạn suy nghĩ dài trước khi trả lời và bỏ qua
`think: false`. Ghép với `format` (JSON schema) thì phần suy nghĩ ăn hết `num_predict` và câu trả lời rỗng (mục 2 lý
thuyết). Cách dùng ở đây: thêm một message `assistant` có nội dung `"{"` ở **cuối** danh sách (prefill). Ollama coi đó là
phần đầu câu trả lời và để model viết tiếp, nên model bỏ qua bước suy nghĩ và tiếp tục ngay một object JSON. Đổi lại,
không còn constrained decoding: JSON có thể hỏng, nên phải tự kiểm.

Viết:

1. `resize_for_vlm(img, max_side)`: `max_side=None` hoặc ảnh đã nhỏ hơn thì trả nguyên ảnh; ngược lại thu nhỏ **giữ tỉ
   lệ** sao cho cạnh dài nhất bằng `max_side` (cạnh kia làm tròn, tối thiểu 1), lọc `Image.LANCZOS`. Không phóng to.
2. `encode_jpeg_b64(img, quality=90) -> str`: chuyển về RGB, lưu JPEG vào bộ nhớ, trả chuỗi base64 (ASCII).
3. `vlm_messages(prompt, image_b64)`: đúng hai message theo kiểu Ollama:
   `{"role": "user", "content": prompt, "images": [image_b64]}` rồi `{"role": "assistant", "content": "{"}`.
4. `parse_reply(text) -> dict | None`: `text` là phần model viết **tiếp sau** `"{"` (có thể model vẫn tự viết lại `{` hoặc
   bọc trong ```` ```json ````). Bỏ khung ```` ``` ```` nếu có, thêm `"{"` vào đầu nếu chuỗi (đã `strip`) chưa bắt đầu bằng
   nó, cắt tới dấu `}` cuối cùng, `json.loads`. Trả `None` khi không parse được, kết quả không phải dict, hoặc `items`
   không phải list. Bỏ các phần tử của `items` không phải dict.
5. `ocr_lines(gt) -> str`: dựng lại văn bản OCR của hoá đơn từ nhãn CORD (`gt["valid_line"]`, mỗi phần tử có `words`,
   mỗi word có `text`, `row_id`, `quad` với `x1..x4`, `y1..y4`). Gom word theo `row_id`; sắp các dòng theo **tâm dọc nhỏ
   nhất** của các word trong dòng (tâm dọc một word là $$(y_1 + y_3)/2$$); trong một dòng sắp word theo `x1`; nối word bằng
   một dấu cách, nối dòng bằng `\n`. Đây là "OCR hoàn hảo" (do người gán nhãn), dùng làm cận trên cho pipeline OCR.

In [ ]:
PROMPT = (
    "You read shop receipts. List every purchased line item and the grand total.\n"
    'Return JSON only: {"items": [{"name": string, "quantity": number, "amount": string}], "total": string}.\n'
    "- amount is the line total exactly as printed (keep the dots and commas).\n"
    "- quantity is 1 if not printed.\n"
    "- Do not list subtotal, tax, service charge, discount, payment, cash or change lines as items.\n"
    "- total is the final amount to pay, exactly as printed."
)
SCHEMA = {
    "type": "object",
    "required": ["items", "total"],
    "properties": {
        "items": {"type": "array", "items": {"type": "object", "required": ["name", "quantity", "amount"], "properties": {
            "name": {"type": "string"}, "quantity": {"type": "number"}, "amount": {"type": "string"}}}},
        "total": {"type": "string"},
    },
}


def resize_for_vlm(img: Image.Image, max_side: int | None) -> Image.Image:
    """Thu nhỏ giữ tỉ lệ cho cạnh dài nhất bằng max_side; None hoặc ảnh đã nhỏ hơn thì giữ nguyên."""
    # TODO
    raise NotImplementedError


def encode_jpeg_b64(img: Image.Image, quality: int = 90) -> str:
    """Ảnh RGB -> JPEG trong bộ nhớ -> base64 ASCII."""
    # TODO
    raise NotImplementedError


def vlm_messages(prompt: str, image_b64: str) -> list[dict]:
    """Message Ollama: user (prompt + ảnh), rồi assistant prefill "{"."""
    # TODO
    raise NotImplementedError


def parse_reply(text: str) -> dict | None:
    """Phần model viết sau prefill "{" -> dict có "items" là list, hoặc None."""
    # TODO
    raise NotImplementedError


def ocr_lines(gt: dict) -> str:
    """Văn bản OCR từ nhãn CORD: gom theo row_id, dòng theo tâm dọc nhỏ nhất, word theo x1."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: nạp tập test (100 hoá đơn), xem một hoá đơn, nhãn và văn bản OCR dựng lại
import glob

import httpx
import pandas as pd

OLLAMA = "http://localhost:11434"
VLM, LLM = "qwen3-vl:2b", "qwen2.5:3b"
N_RECEIPTS = 100   # CPU không có GPU: giảm xuống 20 và chỉ chạy MAX_SIDES = [768]


def load_split(split: str) -> list[tuple[Image.Image, dict]]:
    df = pd.read_parquet(glob.glob(str(CORD / "data" / f"{split}-*.parquet"))[0])
    return [(Image.open(io.BytesIO(r["image"]["bytes"])).convert("RGB"), json.loads(r["ground_truth"]))
            for _, r in df.iterrows()]


test = load_split("test")[:N_RECEIPTS]
golds = [gold_from_cord(gt["gt_parse"]) for _, gt in test]
img, gt = test[2]
print("kích thước ảnh gốc:", img.size)
print(json.dumps(gt["gt_parse"], ensure_ascii=False, indent=1)[:600])
print("nhãn đã chuẩn hoá:", golds[2])
print("--- OCR dựng lại ---\n" + ocr_lines(gt))
sizes = pd.Series([max(im.size) for im, _ in test])
print("cạnh dài nhất của ảnh gốc: p10/p50/p90 =", sizes.quantile([0.1, 0.5, 0.9]).tolist())

In [ ]:
# explore: gọi VLM ở 4 mức độ phân giải, cache câu trả lời (chạy lại không gọi lại model)
MAX_SIDES = [None, 1024, 768, 512]


def ollama_chat(payload: dict) -> tuple[dict, float]:
    t0 = time.perf_counter()
    reply = httpx.post(f"{OLLAMA}/api/chat", json=payload, timeout=1800).json()
    return reply, time.perf_counter() - t0


def run_vlm(max_side) -> list[dict]:
    path = CACHE / f"vlm_{max_side or 'full'}.jsonl"
    rows = [json.loads(line) for line in open(path)] if path.exists() else []
    with open(path, "a") as f:
        for i, (image, _) in enumerate(test):
            if i < len(rows):
                continue
            small = resize_for_vlm(image, max_side)
            reply, dt = ollama_chat({"model": VLM, "stream": False, "keep_alive": "5m",
                                     "messages": vlm_messages(PROMPT, encode_jpeg_b64(small)),
                                     "options": {"temperature": 0, "num_predict": 1500}})
            row = {"i": i, "size": small.size, "tokens": reply.get("prompt_eval_count"), "latency": dt,
                   "text": reply["message"]["content"]}
            rows.append(row)
            f.write(json.dumps(row) + "\n")
    return rows[: len(test)]


def summarize(name: str, rows: list[dict]) -> dict:
    preds = [parse_reply(r["text"]) for r in rows]
    agg = aggregate([score_receipt(p, g) for p, g in zip(preds, golds)])
    lat = pd.Series([r["latency"] for r in rows])
    return {"cấu hình": name, "json_hỏng": sum(p is None for p in preds), "tổng đúng": round(agg["total_acc"], 3),
            "P dòng": round(agg["item_precision"], 3), "R dòng": round(agg["item_recall"], 3),
            "F1 dòng": round(agg["item_f1"], 3), "SL đúng": round(agg["qty_acc"], 3),
            "token TB": round(pd.Series([r.get("tokens") or 0 for r in rows]).mean()),
            "giây p50": round(lat.median(), 2), "giây p95": round(lat.quantile(0.95), 2)}


vlm_runs = {}
for max_side in MAX_SIDES:
    t0 = time.perf_counter()
    vlm_runs[max_side] = run_vlm(max_side)
    print(f"max_side={max_side}: {time.perf_counter() - t0:.0f} s")
httpx.post(f"{OLLAMA}/api/generate", json={"model": VLM, "keep_alive": 0})
table = pd.DataFrame([summarize(f"VLM, cạnh dài {m or 'gốc'}", vlm_runs[m]) for m in MAX_SIDES])
print(table.to_string(index=False))

In [ ]:
# explore: pipeline hai bước: văn bản OCR -> qwen2.5:3b với JSON schema (constrained decoding)
#   (a) OCR hoàn hảo từ nhãn CORD (cận trên)  (b) VLM chép lại chữ trên hoá đơn rồi LLM văn bản trích xuất
OCR_SYSTEM = PROMPT.replace("You read shop receipts.", "You read the OCR text of a shop receipt.")


def run_vlm_transcribe() -> list[str]:
    path = CACHE / "vlm_transcribe.jsonl"
    rows = [json.loads(line) for line in open(path)] if path.exists() else []
    with open(path, "a") as f:
        for i, (image, _) in enumerate(test):
            if i < len(rows):
                continue
            messages = [{"role": "user", "content": "Transcribe all text on this receipt, line by line, exactly as printed.",
                         "images": [encode_jpeg_b64(image)]},
                        {"role": "assistant", "content": "Receipt text:\n"}]
            reply, dt = ollama_chat({"model": VLM, "stream": False, "keep_alive": "5m", "messages": messages,
                                     "options": {"temperature": 0, "num_predict": 1500}})
            row = {"i": i, "latency": dt, "text": reply["message"]["content"]}
            rows.append(row)
            f.write(json.dumps(row) + "\n")
    httpx.post(f"{OLLAMA}/api/generate", json={"model": VLM, "keep_alive": 0})
    return rows[: len(test)]


def run_text_llm(name: str, texts: list[str]) -> list[dict]:
    path = CACHE / f"llm_{name}.jsonl"
    rows = [json.loads(line) for line in open(path)] if path.exists() else []
    with open(path, "a") as f:
        for i, text in enumerate(texts):
            if i < len(rows):
                continue
            messages = [{"role": "system", "content": OCR_SYSTEM}, {"role": "user", "content": "Receipt OCR text:\n" + text}]
            reply, dt = ollama_chat({"model": LLM, "stream": False, "keep_alive": "5m", "messages": messages,
                                     "format": SCHEMA, "options": {"temperature": 0}})
            row = {"i": i, "tokens": reply.get("prompt_eval_count"), "latency": dt, "text": reply["message"]["content"]}
            rows.append(row)
            f.write(json.dumps(row) + "\n")
    return rows[: len(texts)]


transcripts = run_vlm_transcribe()
ocr_gold = run_text_llm("goldocr", [ocr_lines(gt) for _, gt in test])
ocr_vlm = run_text_llm("vlmocr", [r["text"] for r in transcripts])
httpx.post(f"{OLLAMA}/api/generate", json={"model": LLM, "keep_alive": 0})
for rows, t in zip(ocr_vlm, transcripts):
    rows["latency_total"] = rows["latency"] + t["latency"]
two_step = pd.DataFrame([summarize("OCR nhãn + qwen2.5:3b", ocr_gold), summarize("VLM chép chữ + qwen2.5:3b", ocr_vlm)])
print(pd.concat([table, two_step]).to_string(index=False))
print("độ trễ hai bước (VLM chép chữ + LLM), p50:",
      round(pd.Series([r["latency_total"] for r in ocr_vlm]).median(), 2), "s")

In [ ]:
# explore: đọc tận mắt các hoá đơn sai tổng ở cấu hình ảnh gốc
best = [parse_reply(r["text"]) for r in vlm_runs[None]]
wrong = [i for i, (p, g) in enumerate(zip(best, golds)) if score_receipt(p, g)["total_ok"] is False]
print("sai tổng:", wrong)
for i in wrong[:6]:
    p = best[i] or {}
    print(f"\n#{i} ảnh {test[i][0].size} | dự đoán tổng {p.get('total')!r} | nhãn {golds[i]['total']}")
    print("   dự đoán:", [(it.get("name"), it.get("amount")) for it in p.get("items", [])][:6])
    print("   nhãn   :", [(it["name"], it["amount"]) for it in golds[i]["items"]][:6])

## Đọc kết quả

- Bảng VLM: khi giảm cạnh dài từ ảnh gốc xuống 512, số token ảnh giảm bao nhiêu lần, độ trễ giảm bao nhiêu, và trường nào
  (tổng hay dòng hàng) hỏng trước? Chênh lệch nào lớn hơn nhiễu của 100 hoá đơn (một hoá đơn là 0.01 ở cột tổng)?
- Pipeline hai bước với OCR hoàn hảo là **cận trên** của mọi pipeline OCR: VLM end-to-end cách nó bao xa? Pipeline hai
  bước dùng chính VLM để chép chữ thì sao, và tốn thời gian gấp mấy lần?
- Trong các hoá đơn sai tổng, model đọc nhầm số, lấy nhầm dòng (tiền khách đưa, tạm tính), hay nhãn có vấn đề?
- Bộ dữ liệu là hoá đơn Indonesia. Trước khi đưa vào trợ lý tài chính cho người Việt, bạn sẽ đo lại trên gì (mục 2 lý thuyết
  nói cách đổi sang MC-OCR)?

Sau đó làm phần trắc nghiệm của bài trong app.